# 14. 딕셔너리

기초 · 문제 066–070

**사용법**
1. 바로 아래 **🔧 채점기 준비** 셀을 먼저 실행합니다(▶).
2. 각 문제의 `%%writefile` 셀에 코드를 작성하고 실행하면 파일로 저장됩니다.
3. 그 아래 `check(번호)` 셀을 실행하면 채점됩니다.
4. 직접 입력을 넣어 실행해 보려면 새 셀에 `%run p001.py`처럼 입력합니다.

> 정답과 해설은 위키독스 본문의 접힌 상자에 있습니다. 충분히 고민한 뒤에 펼쳐 보세요.

In [ ]:
#@title 🔧 채점기 준비 (먼저 실행하세요)
TESTS = {66: [['the cat and the hat and the bat', 'the 3\nand 2\nbat 1\ncat 1\nhat 1', True, None], ['a', 'a 1', False, None], ['b a b a', 'a 2\nb 2', False, None], ['z y x w', 'w 1\nx 1\ny 1\nz 1', False, None], ['go go go go', 'go 4', False, None]], 67: [['3\nkim 010-0000-1111\nlee 010-0000-2222\nkim 010-0000-3333\n3\nkim\npark\nlee', '010-0000-3333\n없음\n010-0000-2222', True, None], ['1\na 1\n1\nb', '없음', False, None], [None, 'sha256:3a575ed8a6ef9992eb9877165d294845f97f1a15b4b7e651ebf6994e03ac95c9', False, "import random\nrandom.seed(6)\nn = 100000\nprint(n)\nnames = [f'u{random.randrange(60000)}' for _ in range(n)]\nfor x in names:\n    print(x, f'010-{random.randrange(10**4):04d}-{random.randrange(10**4):04d}')\nprint(n)\nfor _ in range(n):\n    print(f'u{random.randrange(80000)}')"]], 68: [['SOS', '... --- ...', True, None], ['HELLO WORLD', '.... . .-.. .-.. --- / .-- --- .-. .-.. -..', True, None], ['E', '.', False, None], ['WHAT HATH GOD WROUGHT', '.-- .... .- - / .... .- - .... / --. --- -.. / .-- .-. --- ..- --. .... -', False, None], ['THE QUICK BROWN FOX JUMPS OVER THE LAZY DOG', '- .... . / --.- ..- .. -.-. -.- / -... .-. --- .-- -. / ..-. --- -..- / .--- ..- -- .--. ... / --- ...- . .-. / - .... . / .-.. .- --.. -.-- / -.. --- --.', False, None]], 69: [['6\nlisten silent evil live vile enlist', '2 3', True, None], ['1\na', '1 1', False, None], ['3\nab ba ab', '1 3', False, None], ['4\nabc def ghi jkl', '4 1', False, None], [None, '83 5665', False, "import random\nrandom.seed(7)\nn = 100000\nprint(n)\nprint(*[''.join(random.choice('abc') for _ in range(random.randint(1, 6))) for _ in range(n)])"]], 70: [['5\nkim math 90\nlee math 85\nkim eng 81\nlee eng 70\nkim sci 100', 'kim 90.3\nlee 77.5', True, None], ['1\nz art 0', 'z 0.0', False, None], ['3\nb x 1\na x 2\nb y 2', 'a 2.0\nb 1.5', False, None], ['6\nm a 100\nm b 99\nm c 99\nn a 1\nn b 0\nn c 0', 'm 99.3\nn 0.3', False, None]]}
TL = {66: 2, 67: 2, 68: 2, 69: 2, 70: 2}
import subprocess, sys, pathlib, hashlib

_GEN_CACHE = {}

def _norm(s):
    return "\n".join(l.rstrip() for l in s.strip("\n").splitlines()).strip("\n")

def _input_of(inp, gen):
    if gen is None:
        return inp
    if gen not in _GEN_CACHE:
        _GEN_CACHE[gen] = subprocess.run([sys.executable, "-c", gen], capture_output=True, text=True).stdout
    return _GEN_CACHE[gen]

def check(pid):
    """p{번호}.py 를 모든 테스트로 실행해 채점합니다."""
    path = pathlib.Path(f"p{pid:03d}.py")
    if not path.exists():
        print(f"먼저 위의 %%writefile p{pid:03d}.py 셀을 실행하세요."); return
    limit = TL[pid]
    ok = 0
    for i, (inp, exp, shown, gen) in enumerate(TESTS[pid], 1):
        tag = f"예제 {i}" if shown else f"숨은 테스트 {i}"
        data = _input_of(inp, gen)
        try:
            r = subprocess.run([sys.executable, str(path)], input=data.rstrip("\n") + "\n",
                               capture_output=True, text=True, timeout=limit)
        except subprocess.TimeoutExpired:
            print(f"⏱ {tag}: 시간 초과 ({limit}초)"); continue
        got = _norm(r.stdout)
        if exp.startswith("sha256:"):
            got_cmp = "sha256:" + hashlib.sha256(got.encode()).hexdigest()
        else:
            got_cmp = got
        if r.returncode:
            last = r.stderr.strip().splitlines()[-1] if r.stderr.strip() else "알 수 없는 오류"
            print(f"💥 {tag}: 실행 오류 → {last}")
        elif got_cmp == exp:
            ok += 1; print(f"✅ {tag}")
        elif shown:
            print(f"❌ {tag}\n   입력: {inp!r}\n   기대: {exp!r}\n   실제: {got!r}")
        elif gen is not None:
            print(f"❌ {tag}  (큰 입력)")
        else:
            print(f"❌ {tag}  (입력: {inp[:60]!r})")
    total = len(TESTS[pid])
    print("─" * 30)
    print(f"🎉 {pid:03d}번 통과! ({ok}/{total})" if ok == total else f"{ok}/{total} 통과 — 다시 도전해 보세요.")
print('채점기 준비 완료!')

## 066. 단어의 빈도

영어 소문자 단어들이 공백으로 구분되어 한 줄로 주어집니다. 각 단어의 등장 횟수를 **횟수가 많은 순서로**, 같으면 단어의 사전 순서로, 한 줄에 하나씩 `단어 횟수` 형식으로 출력하십시오.

**예제 1**
입력
```
the cat and the hat and the bat
```
출력
```
the 3
and 2
bat 1
cat 1
hat 1
```

In [ ]:
%%writefile p066.py
# 여기에 코드를 작성하세요


In [ ]:
check(66)

## 067. 전화번호부

첫째 줄에 N, 이어서 N줄에 `이름 번호`가 주어집니다. 다음 줄에 질의 수 Q, 이어서 Q줄에 이름이 주어집니다. 각 질의마다 번호를 출력하되, 없는 이름이면 `없음`을 출력하십시오. 같은 이름이 두 번 등록되면 **나중 번호**가 유효합니다.

**제한** 1 ≤ N, Q ≤ 10⁵

**예제 1**
입력
```
3
kim 010-0000-1111
lee 010-0000-2222
kim 010-0000-3333
3
kim
park
lee
```
출력
```
010-0000-3333
없음
010-0000-2222
```

In [ ]:
%%writefile p067.py
# 여기에 코드를 작성하세요


In [ ]:
check(67)

## 068. 모스 부호

영어 대문자와 공백으로 된 문장을 국제 모스 부호로 바꾸어 출력하십시오. 글자 사이는 공백 하나, 단어 사이는 ` / `(공백-빗금-공백)로 구분합니다. A부터 Z까지의 부호는 다음과 같습니다(복사해서 쓰십시오).

```
.- -... -.-. -.. . ..-. --. .... .. .--- -.- .-.. -- -. --- .--. --.- .-. ... - ..- ...- .-- -..- -.-- --..
```

**예제 1**
입력
```
SOS
```
출력
```
... --- ...
```

**예제 2**
입력
```
HELLO WORLD
```
출력
```
.... . .-.. .-.. --- / .-- --- .-. .-.. -..
```

In [ ]:
%%writefile p068.py
# 여기에 코드를 작성하세요


In [ ]:
check(68)

## 069. 아나그램 모임

첫째 줄에 N, 둘째 줄에 N개의 영어 소문자 단어가 주어집니다. 서로 아나그램인 단어끼리 한 모임으로 묶을 때, **모임의 수**와 **가장 큰 모임의 크기**를 공백으로 구분해 출력하십시오. (같은 단어가 두 번 나오면 같은 모임에 두 번 들어갑니다.)

**제한** 1 ≤ N ≤ 10⁵, 단어 길이 ≤ 10

**예제 1**
입력
```
6
listen silent evil live vile enlist
```
출력
```
2 3
```

In [ ]:
%%writefile p069.py
# 여기에 코드를 작성하세요


In [ ]:
check(69)

## 070. 학생별 평균

첫째 줄에 N, 이어서 N줄에 `이름 과목 점수`가 주어집니다. 학생 이름의 사전 순서로, 각 학생의 평균 점수를 소수점 아래 첫째 자리까지 `이름 평균` 형식으로 출력하십시오.

**제한** 1 ≤ N ≤ 1000, 0 ≤ 점수 ≤ 100, 한 학생의 과목 수는 3개 이하

**예제 1**
입력
```
5
kim math 90
lee math 85
kim eng 81
lee eng 70
kim sci 100
```
출력
```
kim 90.3
lee 77.5
```

In [ ]:
%%writefile p070.py
# 여기에 코드를 작성하세요


In [ ]:
check(70)